# Track D / Day 2 — Ghost author generation (Colab)

Implements `pilot_0_1_execution_spec.md` §2.2 step 2, using the real Day 1 outputs already committed to `shravanidhus31/unlearning-audit-study` (`ghosts/schema.json`, `exemplars.json`, `length_stats.json`).

**Generator: mixed — Groq + Anthropic (`docs/DEVIATIONS.md` D-005).** 14 of 30 authors already have real, valid data from Groq (`openai/gpt-oss-120b`, D-004), generated before Groq's daily token quota and a recurring JSON-validation error made finishing the rest impractical. Rather than discard that real data, the remaining ~16 authors are backfilled with the Anthropic API (`claude-opus-5`) now that the account is funded — the final ghost set is a deliberate, documented mix of two generators, traceable per-author via each checkpoint's `_meta.provider`/`_meta.model`. Gemini (`gemini-3.6-flash`, D-003) was tried before Groq but its real free quota (20 requests/day) was too small. All three provider code paths (`--provider anthropic|gemini|groq`) remain in the script.

**Before running:** add a Colab secret (key icon in the left sidebar) —
- `ANTHROPIC_API_KEY` — from [console.anthropic.com](https://console.anthropic.com); this is now a **paid** key (unlike the earlier free Gemini/Groq keys), so real API cost applies from here on.
- `GH_TOKEN` — only needed if you want this notebook to `git pull`/read a private repo; skip if the repo is public and you'll commit manually afterward (the chosen persistence mode: Drive checkpointing, manual commit).

This notebook never pushes to GitHub itself. Everything is checkpointed to Drive as it runs; you review and `git add/commit/push` yourself at the end.

## 1. Mount Drive and clone the repo into it
Cloning straight into Drive means every checkpoint the script writes is already persisted — a Colab disconnect costs minutes, not the run, per the spec's own §3 risk note.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/unlearning_pilot'
REPO_DIR = os.path.join(PROJECT_DIR, 'unlearning-audit-study')
os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shravanidhus31/unlearning-audit-study.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log -1 --oneline

## 2. Add `day2_generate.py` to the repo's `scripts/` folder
Paste in the version reviewed alongside this notebook (`scripts/day2_generate.py`). If you've already committed it to the repo, this cell is a no-op — the `git pull` above already has it.

In [ ]:
import os
assert os.path.exists('scripts/day2_generate.py'), (
    "scripts/day2_generate.py not found in the cloned repo.\n"
    "Upload it into this Colab session at that path (Files pane, drag-and-drop\n"
    "into scripts/), or git-commit it to the repo first and re-run cell 1."
)
print('day2_generate.py present.')

## 3. Install dependencies

In [ ]:
!pip install -q anthropic openai google-genai pandas numpy scipy transformers datasets

## 4. Secrets

In [ ]:
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('ANTHROPIC_API_KEY loaded:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## 5. Goodreads CSV (included by decision — not a spec requirement)
Needs a Kaggle account + API token (`kaggle.json`). Upload yours if `kagglehub` prompts for it, or upload `data/books.csv` manually via the Files pane and skip this cell.

In [ ]:
GOODREADS_CSV = os.path.join(REPO_DIR, 'data', 'books.csv')
os.makedirs(os.path.dirname(GOODREADS_CSV), exist_ok=True)

if not os.path.exists(GOODREADS_CSV):
    !pip install -q kagglehub
    import kagglehub, shutil
    path = kagglehub.dataset_download('jealousleopard/goodreadsbooks')
    src = os.path.join(path, 'books.csv')
    shutil.copy(src, GOODREADS_CSV)

print('Goodreads CSV at:', GOODREADS_CSV, '-', os.path.exists(GOODREADS_CSV))

## 6. Self-test (offline — no API calls, sanity-check before spending anything)

In [ ]:
!python scripts/day2_generate.py --selftest

## 7. Dry run — read the actual prompt before spending API credits
Prints author 0's full prompt with no API call. Read it end to end once.

In [ ]:
!python scripts/day2_generate.py --provider groq --dry-run --authors 0 \
    --goodreads {GOODREADS_CSV} --outdir ghosts

## 8. Pilot — 5 authors, never all 30 blind
Checks Cohen's d / KS p against holdout10 automatically at the end. Read the verdict before continuing to the full run.

In [ ]:
!python scripts/day2_generate.py --provider groq --authors 0-4 \
    --goodreads {GOODREADS_CSV} --outdir ghosts

In [ ]:
# Read the pilot verdict straight from the log rather than re-deriving it.
with open('ghosts/generation_log.md', encoding='utf-8') as f:
    log = f.read()
start = log.find('## 6. Length check')
end = log.find('## 7.')
print(log[start:end])

**Stop here if the verdict above says STOP.** Fix the length instruction in `scripts/day2_generate.py`'s `build_prompt()` (or the length target itself) and re-run the pilot — this is a logged deviation, not a silent edit, per `docs/DEVIATIONS.md` conventions. Only continue to the full run once the pilot says PROCEED.

## 9. Backfill remaining authors with Anthropic (D-005) — real API cost from here

14 authors (0,1,2,3,4,5,6,7,9,12,13,14,15,16) already have real, valid Groq data and will be skipped automatically by `--resume`. This call only generates the ~16 still missing/failed (8,10,11,17,18-29). Estimated cost: roughly $1.50-2 for ~16 authors at Claude Opus 5 pricing (see chat discussion) — watch your [usage page](https://platform.claude.com/usage) if you want to track it live.

In [ ]:
# Optional but recommended: one dry-run under Anthropic before spending real
# money. The prompt itself doesn't change by provider (already verified
# multiple times), but this confirms the API key resolves correctly first.
!python scripts/day2_generate.py --provider anthropic --dry-run --authors 8 \
    --goodreads {GOODREADS_CSV} --outdir ghosts

In [ ]:
!python scripts/day2_generate.py --provider anthropic --authors 0-29 --resume \
    --goodreads {GOODREADS_CSV} --outdir ghosts

In [ ]:
import json
rows = [json.loads(l) for l in open('ghosts/candidates_raw.jsonl', encoding='utf-8')]
authors = {r['author_id'] for r in rows}
print(f'{len(rows)} QA rows across {len(authors)} authors (target: 600 rows / 30 authors)')
if len(rows) != 600 or len(authors) != 30:
    print('NOT COMPLETE YET — re-run cell 9 with --resume for the missing author_ids.')
else:
    print('600/600. Ready for Day 3 (collision filter, spec §2.2 step 3).')

# Per-generator breakdown -- the ghost set is a deliberate Groq+Anthropic mix (D-005).
by_provider = {}
for aid in authors:
    ckpt = json.load(open(f'ghosts/checkpoints/author_{aid:02d}.json', encoding='utf-8'))
    prov = ckpt.get('_meta', {}).get('provider', 'unknown')
    by_provider[prov] = by_provider.get(prov, 0) + 1
print('Authors by generator:', by_provider)

print()
print('To commit manually:')
print('  git add ghosts/candidates_raw.jsonl ghosts/generation_log.md ghosts/checkpoints/')
print('  git commit -m "Track D Day 2: generate 600 ghost candidates (Groq + Anthropic, D-005)"')
print('  git push')